In [1]:
import os
P8_BUCKET = os.environ["P8_BUCKET"]

In [2]:
#q1
with open("/etc/os-release") as f:
    data = f.read()
data

'PRETTY_NAME="Ubuntu 24.04.4 LTS"\nNAME="Ubuntu"\nVERSION_ID="24.04"\nVERSION="24.04.4 LTS (Noble Numbat)"\nVERSION_CODENAME=noble\nID=ubuntu\nID_LIKE=debian\nHOME_URL="https://www.ubuntu.com/"\nSUPPORT_URL="https://help.ubuntu.com/"\nBUG_REPORT_URL="https://bugs.launchpad.net/ubuntu/"\nPRIVACY_POLICY_URL="https://www.ubuntu.com/legal/terms-and-policies/privacy-policy"\nUBUNTU_CODENAME=noble\nLOGO=ubuntu-logo\n'

In [3]:
#q2
import subprocess

result = subprocess.run(["lscpu"], capture_output=True, text=True)
result.stdout

'Architecture:                            x86_64\nCPU op-mode(s):                          32-bit, 64-bit\nAddress sizes:                           46 bits physical, 48 bits virtual\nByte Order:                              Little Endian\nCPU(s):                                  2\nOn-line CPU(s) list:                     0,1\nVendor ID:                               GenuineIntel\nModel name:                              Intel(R) Xeon(R) CPU @ 2.20GHz\nCPU family:                              6\nModel:                                   79\nThread(s) per core:                      2\nCore(s) per socket:                      1\nSocket(s):                               1\nStepping:                                0\nBogoMIPS:                                4400.48\nFlags:                                   fpu vme de pse tsc msr pae mce cx8 apic sep mtrr pge mca cmov pat pse36 clflush mmx fxsr sse sse2 ss ht syscall nx pdpe1gb rdtscp lm constant_tsc rep_good nopl xtopology nonstop_tsc cpuid

In [4]:
import pyarrow.fs as fs

gcs = fs.GcsFileSystem()
bucket = P8_BUCKET

with open("wi-schools-raw.parquet", "rb") as f:
    with gcs.open_output_stream(f"{bucket}/wi-schools-raw.parquet") as out:
        out.write(f.read())

In [5]:
#q3
import pyarrow.fs as fs

gcs = fs.GcsFileSystem()

bucket = P8_BUCKET

# list top-level contents
info = gcs.get_file_info(fs.FileSelector(f"{bucket}/", recursive=False))

# extract paths
paths = [item.path for item in info]
paths

['cs544-s26-bucket/wi-schools-raw.parquet']

In [6]:
import os
os.getcwd()

'/home/nwu/p8_nautomo/src'

In [7]:
from google.cloud import dataform

dfm = dataform.DataformClient()

PROJECT = "cs544-491017"
REPO = "cs544-s26"
WORKSPACE = "cs544-s26"

parent = f"projects/{PROJECT}/locations/us-central1/repositories/{REPO}"

workspace = f"{parent}/workspaces/{WORKSPACE}"

dfm.write_file({
    "workspace": workspace,
    "path": "definitions/wi_counties.sqlx",
    "contents": open("definitions/wi_counties.sqlx").read().encode()
})

dfm.write_file({
    "workspace": workspace,
    "path": "definitions/schools.sqlx",
    "contents": open("definitions/schools.sqlx").read().encode()
})

dfm.write_file({
    "workspace": workspace,
    "path": "definitions/wi_county_schools.sqlx",
    "contents": open("definitions/wi_county_schools.sqlx").read().encode()
})

In [8]:
#q4
import pyarrow.fs as fs

gcs = fs.GcsFileSystem()

info = gcs.get_file_info(f"{P8_BUCKET}/wi-schools-raw.parquet")

info.mtime_ns

1777684359609000000

In [9]:
compilation = dfm.create_compilation_result(dataform.CreateCompilationResultRequest(
    parent="projects/cs544-491017/locations/us-central1/repositories/cs544-s26",
    compilation_result=dataform.CompilationResult(
        workspace="projects/cs544-491017/locations/us-central1/repositories/cs544-s26/workspaces/cs544-s26",
    ),
))
#compilation.compilation_errors
assert compilation.compilation_errors == []

In [10]:
#q5
from google.cloud import dataform

df_client = dataform.DataformClient()

compilation_result_name = compilation.name

response = df_client.query_compilation_result_actions(
    request={"name": compilation_result_name}
)

deps = {}

for action in response.compilation_result_actions:
    name = action.target.name
    
    dependencies = [
        dep.name for dep in action.relation.dependency_targets
    ] if action.relation else []
    
    deps[name] = dependencies


deps

{'schools': [],
 'wi_counties': [],
 'wi_county_schools': ['schools', 'wi_counties']}

In [11]:
from google.cloud import bigquery

bq = bigquery.Client()

dataset_id = "cs544-491017.p8_nautomo"

dataset = bigquery.Dataset(dataset_id)
dataset.location = "US"

bq.create_dataset(dataset, exists_ok=True)

Dataset(DatasetReference('cs544-491017', 'p8_nautomo'))

In [12]:
#q6
from google.cloud import bigquery

bq = bigquery.Client()

query = """
SELECT COUNT(*) AS num_counties
FROM `cs544-491017.p8_nautomo.wi_counties`
"""

bq.query(query).to_dataframe()

NotFound: 404 Not found: Table cs544-491017:p8_nautomo.wi_counties was not found in location US; reason: notFound, message: Not found: Table cs544-491017:p8_nautomo.wi_counties was not found in location US

Location: US
Job ID: 8451a50b-9f1d-45aa-9ea4-37749b33d372
